# Relatório Projeto Aplicado - Smart Crop Irrigation Grid: Avaliação de Algoritmos de RL Inteligente

**Integrantes do Grupo:**
* Gregory Stein
* Jackson Pandolfo
* Lucas Toledo

Neste projeto, avaliamos três algoritmos de Aprendizado por Reforço Profundo (PPO, DQN e A2C) na resolução de um problema logístico em agricultura de precisão. O objetivo é treinar um robô autônomo para gerenciar a umidade de uma plantação em um grid 5x5, regando quando necessário, colhendo no momento ideal e entregando os itens na base.

**Visão Geral e Descobertas:**
A dificuldade central do ambiente é a **esparsidade de recompensas**: o agente perde pontos a cada passo e a cada planta morta, mas só recebe a pontuação ótima (+20) ao concluir toda a cadeia de passos (regar -> colher -> navegar até a base -> descarregar). Nossos experimentos revelaram que algoritmos *off-policy* (como o DQN) lidam melhor com esse cenário graças à memória de longo prazo (*Replay Buffer*). Em contrapartida, algoritmos *on-policy* (PPO e A2C) sofrem de "miopia de recompensa" e gestão ineficiente de inventário, exigindo uma técnica rigorosa de *Reward Shaping* (punições por inatividade e colisão) para resolver o problema com sucesso.

In [ ]:
!pip install stable-baselines3[extra]

In [ ]:
# Importações base e verificação do ambiente
import gymnasium as gym
from stable_baselines3 import PPO, DQN, A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from smart_crop_irrigation_V2 import SmartCropIrrigationEnv

# Instanciando o ambiente com punições ativas (Reward Shaping)
env = SmartCropIrrigationEnv(
    grid_size=5,
    max_steps=250,
    dry_rate=0.02,        # Taxa mitigada para estender a janela de sobrevivência
    border_penalty=1.0,   # Punição por colisão
    idle_penalty=0.5      # Punição por inatividade
)
print("Ambiente V2 carregado com sucesso. Espaço de Ação:", env.action_space)

## 1. Modelagem do Problema (MDP)

O ambiente foi modelado como um Processo de Decisão de Markov (MDP) customizado no `gymnasium`, definido pela tupla $(S, A, T, R)$:

* **Estado ($S$):** O espaço de observação é um dicionário (`spaces.Dict`) contendo a posição do robô (x, y), a ocupação do inventário (0 a 3 plantas), uma matriz 2D representando a umidade do grid (níveis de 0 a 4) e um indicador booleano de presença na base.
* **Ações ($A$):** O espaço de ações é discreto (`spaces.Discrete(8)`), permitindo 8 comandos: Mover (Cima, Baixo, Esquerda, Direita), Regar, Colher, Descarregar e Esperar.
* **Transição ($T$):** As movimentações do robô são determinísticas. No entanto, o ambiente possui transições estocásticas baseadas na degradação temporal: a cada passo, há uma probabilidade (`dry_rate`) de as plantas secarem e morrerem.
* **Recompensa ($R$):** A função de recompensa original é esparsa (+20 por entrega, +5 por colheita correta, -0.1 por passo, -2 por morte da planta). Para os testes avançados (Ambiente V2), aplicamos *Reward Shaping* punitivo: -1.0 por tentar sair dos limites do grid e -0.5 por inatividade.

## 2. Metodologia de Experimentos

Para avaliar a viabilidade de controle do robô, implementamos três arquiteturas de redes neurais do pacote `stable_baselines3`:
1. **DQN (Deep Q-Network):** Representando as políticas *off-policy*, utilizando *Replay Buffer* para reaproveitar experiências passadas.
2. **PPO (Proximal Policy Optimization):** Algoritmo *on-policy* padrão, conhecido por sua estabilidade de atualização.
3. **A2C (Advantage Actor-Critic):** Algoritmo *on-policy* clássico e síncrono.

**Métricas e Hiperparâmetros:**
A avaliação do desempenho foi pautada na **Recompensa Média** e na **Taxa de Entregas** (quantas das 24 plantas foram entregues com sucesso antes do limite de 250 passos). Para cada algoritmo, executamos scripts de benchmark (`benchmark_*.py`) testando diferentes configurações de hiperparâmetros (como `learning_rate`, tamanho do `batch_size`, `n_steps` e coeficiente de entropia `ent_coef`) em rodadas curtas de 150.000 passos, selecionando a melhor configuração para o treinamento longo (300k a 1M de passos).

## 3. Avaliação do PPO e DQN

Os primeiros testes estabeleceram a linha de base de dificuldade do ambiente.

**O Desempenho do PPO:** No benchmark de 150k passos, o algoritmo PPO lutou para convergir, mantendo recompensas negativas (melhor configuração `rollout_longo_gamma_alto` atingiu -31.4). Os testes revelaram que a cota de 150k passos era insuficiente para a exploração de um algoritmo *on-policy* sem *Reward Shaping* agressivo.

**O Sucesso do DQN:** Em contraste, o DQN apresentou resiliência logística. Beneficiando-se do *Replay Buffer* (memória de longo prazo), o algoritmo off-policy conseguiu revisitar estados ótimos, atingindo pontuações positivas de **443.6 ± 120.1** na configuração de `exploracao_longa`. O DQN aprendeu a navegar pelo grid e colher e entregar plantas, tendo eficiência para conclusão do objetivo.

In [5]:
# Configuração candidata do PPO (melhor resultado no benchmark de 150k passos)
ppo_params = {
    "learning_rate": 3e-4,
    "n_steps": 4096,
    "batch_size": 64,
    "gamma": 0.995
}

# Demonstração da configuração base do modelo
print("Hiperparâmetros do modelo PPO (rollout_longo_gamma_alto):", ppo_params)

Hiperparâmetros do modelo PPO (rollout_longo_gamma_alto): {'learning_rate': 0.0003, 'n_steps': 4096, 'batch_size': 64, 'gamma': 0.995}


In [3]:
# Configuração vencedora do DQN (treinada por 300.000 passos)
dqn_params = {
    "learning_rate": 0.0005,
    "exploration_fraction": 0.6,
    "batch_size": 64,
    "gamma": 0.99
}

# Demonstração da configuração base do modelo
print("Hiperparâmetros do modelo DQN vencedor:", dqn_params)

Hiperparâmetros do modelo DQN vencedor: {'learning_rate': 0.0005, 'exploration_fraction': 0.6, 'batch_size': 64, 'gamma': 0.99}


## 3.1. Avaliação do A2C
O algoritmo A2C (*Advantage Actor-Critic*), sendo *on-policy* e sem Replay Buffer, foi o que mais sofreu com a formulação inicial do problema.

### O Colapso de Política (Baseline)
Com a taxa de secagem alta (`dry_rate=0.08`) e a versão clássica do ambiente, todas as 6 configurações do A2C travaram exatamente em **-73.0** pontos.

O agente sofreu de **Miopia de Recompensa**: ao explorar e cometer erros, acumulou tantas punições que a rede neural calculou que a estratégia matematicamente mais segura era fugir para o canto do grid e ficar parado, aceitando a penalidade de -73.0 para evitar perdas maiores.

### A Intervenção: Reward Shaping (Ambiente V2) e Mitigação de Sobrevivência
Para forçar o A2C a sair do "ótimo local", foi implementado no `smart_crop_irrigation_V2` penalidades caso o agente permaneça inerte ou nas bordas do grid, junto de uma redução da taxa de secagem das plantas(`dry_rate=0.02`).
* **-1.0** por colisão nas bordas.
* **-0.5** por inatividade.

Essas configurações forçaram o agente a sair e explorar o grid, trazendo resultados constantes.

In [4]:
# A configuração matemática que resolveu o colapso do A2C
# A alta entropia (0.1) aliada ao Reward Shaping V2 permitiu a otimização da rota
a2c_winner_params = {
    "learning_rate": 7e-4,
    "n_steps": 5,
    "gamma": 0.99,
    "ent_coef": 0.1 # Nível Extremo de Exploração
}

print("Hiperparâmetros A2C (Otimização Final):", a2c_winner_params)

Hiperparâmetros A2C (Otimização Final): {'learning_rate': 0.0007, 'n_steps': 5, 'gamma': 0.99, 'ent_coef': 0.1}


## 3.2. Análise Técnica e A2C

Após 300.000 passos com a configuração `exploracao_extrema` e o *Reward Shaping* ativo, o A2C performou de forma muito mais eficaz, comparado com os treinos iniciais:
* **Média no Benchmark:** 314.2 pontos (superando com folga o colapso de -73.0).
* **Capacidade Logística:** O robô compreendeu o inventário. Passou a coletar de 3 em 3 plantas antes de retornar à base.
* **Volume de Entregas:** Atingiu de 15 a 20 plantas entregues por episódio, chegando extremamente próximo à performance ótima do DQN (23 plantas).


## 4. Conclusões

**Visão Geral:** O projeto demonstrou com sucesso que tarefas logísticas complexas e com recompensas esparsas podem ser solucionadas por meio de Aprendizado por Reforço Profundo. O DQN provou ser o algoritmo mais natural para o problema original, enquanto o A2C exigiu modelagem matemática adicional, mas conseguiu atingir eficiência logística semelhante após o ajuste fino.

**Dificuldades Encontradas:** A principal barreira foi o "Colapso de Política" nos algoritmos *on-policy*. O A2C inicialmente sofria de gestão ineficiente de inventário (colhia uma planta e voltava para a base, ignorando a capacidade de carregar 3 itens) e "Policy Churn" (loop de movimentação inútil para evitar perdas matemáticas ao explorar).

**Limitações:** O modelo atual assume a taxa de secagem das plantas (`dry_rate`) de forma global e simplificada, e a matriz de observação, apesar de funcional para um grid 5x5, pode sofrer de *maldição da dimensionalidade* caso o grid seja expandido para tamanhos industriais sem a adição de redes convolucionais (CNNs).

**Trabalhos Futuros:** Para desenvolvimentos posteriores, sugere-se a implementação de um sistema Multi-Agente (onde múltiplos robôs dividem setores do grid), a transição do espaço de ações discreto para um espaço contínuo (movimentação vetorial do robô), e a introdução de obstáculos dinâmicos no ambiente agrícola.

Link da Apresentação: [texto do link](https://)
